In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [66]:
#lấy thêm 1 bộ dữ liệu tương đồng ở trên Kaggle
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nadyinky/sephora-products-and-skincare-reviews")

#print(os.listdir(path))
file_path = os.path.join(path,'product_info.csv')
sepho = pd.read_csv(file_path)

sepho.head(1)





,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.6364,11.0,NaN,NaN,NaN,...,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scen...",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,NaN,NaN


In [67]:
#bộ dữ liệu hiện đang có
phara = pd.read_csv('pharmacity_personal_care.csv').drop(columns=['URL'])
phara.head(1)

,ProductID,ProductName,Category,CategoryCode,SourceSlug,Brand,Price,OriginalPrice,DiscountPercent,Unit,Rating,TotalSold
0,P07609,Lăn Khử Mùi NIVEA Ngọc Trai Quyến Rũ (chai 25ml),Lăn khử mùi,lan-khu-mui,san-pham-khu-mui,Nivea,49500,49500,0.0,Chai,4.75,7423


In [68]:
#inspect dữ liệu của data bên kaggle
print("\nPharmacity columns:")
print(phara.columns.tolist())

print("\nSecond source columns:")
print(sepho.columns.tolist())




Pharmacity columns:
['ProductID', 'ProductName', 'Category', 'CategoryCode', 'SourceSlug', 'Brand', 'Price', 'OriginalPrice', 'DiscountPercent', 'Unit', 'Rating', 'TotalSold']

Second source columns:
['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'rating', 'reviews', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count', 'child_max_price', 'child_min_price']


In [69]:
print("\n=== PHARMACITY DTYPES ===")
print(phara.dtypes)

print("\n=== SECOND SOURCE DTYPES ===")
print(sepho.dtypes)


=== PHARMACITY DTYPES ===
ProductID           object
ProductName         object
Category            object
CategoryCode        object
SourceSlug          object
Brand               object
Price                int64
OriginalPrice        int64
DiscountPercent    float64
Unit                object
Rating             float64
TotalSold            int64
dtype: object

=== SECOND SOURCE DTYPES ===
product_id             object
product_name           object
brand_id                int64
brand_name             object
loves_count             int64
rating                float64
reviews               float64
size                   object
variation_type         object
variation_value        object
variation_desc         object
ingredients            object
price_usd             float64
value_price_usd       float64
sale_price_usd        float64
limited_edition         int64
new                     int64
online_only             int64
out_of_stock            int64
sephora_exclusive       int64
highl

In [70]:
#làm việc với cột của Pharamacity
pharmacity_std = phara.copy()
pharmacity_std["Source"] = "Pharmacity_VN"
pharmacity_std["Currency"] = "VND"

common_col = ["ProductID", "ProductName", "Brand", "Category",
              "Price", "Rating", "Source", "Currency"]
pharmacity_std = pharmacity_std[common_col]
pharmacity_std["Price_VND_ref"] = pharmacity_std["Price"]
#làm việc với cột của Sephora
sephora_std = sepho.copy()
sephora_std["Source"] = "Sephora_US"
sephora_std["Currency"] = "USD"

sephora_std = sephora_std.rename(columns={
    "product_id": "ProductID",
    "product_name": "ProductName",
    "primary_category": "Category",
    "brand_name": "Brand",
    "price_usd": "Price",
    "rating": "Rating",
    "reviews": "Reviews",
})
sephora_std = sephora_std[common_col]
EXCHANGE_RATE_USD_VND = 25_400
sephora_std["Price_VND_ref"] = (sephora_std["Price"] * EXCHANGE_RATE_USD_VND).round(0)

display(pharmacity_std.head(1))
display(sephora_std.head(1))



,ProductID,ProductName,Brand,Category,Price,Rating,Source,Currency,Price_VND_ref
0,P07609,Lăn Khử Mùi NIVEA Ngọc Trai Quyến Rũ (chai 25ml),Nivea,Lăn khử mùi,49500,4.75,Pharmacity_VN,VND,49500


,ProductID,ProductName,Brand,Category,Price,Rating,Source,Currency,Price_VND_ref
0,P473671,Fragrance Discovery Set,19-69,Fragrance,35.0,3.6364,Sephora_US,USD,889000.0


In [71]:
for df in (pharmacity_std, sephora_std):
    df["ProductID"] = df["ProductID"].astype(str)
    df["ProductName"] = df["ProductName"].astype(str)
    df["Brand"] = df["Brand"].astype(str)
    df["Category"] = df["Category"].astype(str)
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
    df["Price"] = df["Price"].astype("float64")
    df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
    df["Source"] = df["Source"].astype(str)
    df["Currency"] = df["Currency"].astype(str)
    df["Price_VND_ref"] = pd.to_numeric(df["Price_VND_ref"], errors="coerce")
    df["Price_VND_ref"] = df["Price_VND_ref"].astype("float64")

# kiem tra lai truoc khi concat
print(pharmacity_std.dtypes)
print(sephora_std.dtypes)

ProductID         object
ProductName       object
Brand             object
Category          object
Price            float64
Rating           float64
Source            object
Currency          object
Price_VND_ref    float64
dtype: object
ProductID         object
ProductName       object
Brand             object
Category          object
Price            float64
Rating           float64
Source            object
Currency          object
Price_VND_ref    float64
dtype: object


In [72]:
master = pd.concat([pharmacity_std, sephora_std], axis=0, ignore_index=True, sort=False)
print("\n=== Bộ Dữ Liệu Tổng Hợp Sau khi Concat ===")
print("Shape:", master.shape)
print(master["Source"].value_counts())
print("\nMissing values per column (dung de dan chung cho Step 4 - imputation):")
print(master.isnull().sum())
 
master.to_csv("unified_static_master_table.csv", index=False)


=== Bộ Dữ Liệu Tổng Hợp Sau khi Concat ===
Shape: (8939, 9)
Source
Sephora_US       8494
Pharmacity_VN     445
Name: count, dtype: int64

Missing values per column (dung de dan chung cho Step 4 - imputation):
ProductID          0
ProductName        0
Brand              0
Category           0
Price              0
Rating           278
Source             0
Currency           0
Price_VND_ref      0
dtype: int64


In [ ]:
##Thực hiện việc merge
transactions = pd.read_csv("pharmacity_simulated_transactions.csv")
transactions_merge = transactions.drop(columns=['Brand','Category','ProductName'])
transactions_merge = transactions_merge.rename(columns={'Price':'UnitPrice'})


df_final = transactions_merge.merge(
    master,
    on='ProductID',
    how='left',
    validate='many_to_one',  
)
df_final



df_final.columns.tolist()
#mismatch = final[final["Price_x"] != final["Price_y"]]
#print(mismatch[["ProductID", "Price_x", "Price_y"]].head())

['OrderID',
 'CustomerID',
 'CustomerName',
 'Age',
 'Gender',
 'ProductID',
 'UnitPrice',
 'Quantity',
 'TotalAmount',
 'PurchaseDate',
 'City',
 'PaymentMethod',
 'ProductName',
 'Brand',
 'Category',
 'Price',
 'Rating',
 'Source',
 'Currency',
 'Price_VND_ref']

Dữ liệu giao dịch chỉ mô phỏng cho Pharmacity, nên 8.494 sản phẩm Sephora không thể có đơn hàng. Đó là dữ liệu không tồn tại theo cấu trúc, không phải dữ liệu bị thiếu.

In [74]:
df_final.to_csv("final_master_dataset.csv", index=False)